# 07. Phenotype Analysis: Epigenetic Age Deviation

Evaluate biological associations of epigenetic age deviation (EAD) across clocks and phenotypes.

**Pipeline:**
1. Compute age deviation residuals from cross-validation predictions (from notebook 06)
2. Fit OLS regressions: EAD ~ phenotype + covariates (age, age^2, sex)
3. Delta-AIC comparison: quadratic vs linear age model for Myeloid clock
4. Forest plot across phenotypes and cell-type clocks

**Inputs:**
- `results/age_references_v2/cross_reference_ont_pooled_testonly_agegrid_0_100/summary_df.csv` (from notebook 06)

**Outputs:**
- OLS regression tables (beta, SE, p-value) per phenotype per clock
- Forest diagram PDF

---

In [ ]:
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.api as sm
from scipy import stats
from google.cloud import storage

WORKSPACE_BUCKET = os.environ['WORKSPACE_BUCKET']
WORKSPACE_CDR    = os.environ['WORKSPACE_CDR']
GOOGLE_PROJECT   = os.environ.get('GOOGLE_PROJECT', '')
BUCKET_NAME      = WORKSPACE_BUCKET.replace('gs://', '')
client           = storage.Client()

CROSS_REF_GCS = (
    f'{WORKSPACE_BUCKET}/results/age_references_v2/'
    'cross_reference_ont_pooled_testonly_agegrid_0_100/summary_df.csv'
)

print('WORKSPACE_BUCKET:', WORKSPACE_BUCKET)
print('CROSS_REF_GCS   :', CROSS_REF_GCS)


In [ ]:
def gs_to_bucket_blob(gs_path):
    assert gs_path.startswith('gs://'), gs_path
    rest = gs_path[5:]
    bkt, obj = rest.split('/', 1)
    return bkt, obj

def download_text_from_gcs(gs_path):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    if not blob.exists():
        return None
    return blob.download_as_text()

import io
def read_csv_from_gcs(gs_path):
    txt = download_text_from_gcs(gs_path)
    if txt is None:
        raise FileNotFoundError(gs_path)
    return pd.read_csv(io.StringIO(txt))


### Step 1: Load age predictions from cross-reference evaluation (notebook 06)

In [ ]:
print('Loading cross-reference evaluation from GCS...')
summary_df = read_csv_from_gcs(CROSS_REF_GCS)

print('summary_df shape:', summary_df.shape)
display(summary_df.head())

# Focus on matched reference-read pairs (same cell type)
matched_df = summary_df[
    summary_df['reference'] == summary_df['read_type']
].copy().reset_index(drop=True)

print('Matched pairs:', len(matched_df))
print('Cell types:', sorted(matched_df['reference'].unique()))


### Step 2: Load clinical phenotype data from BigQuery

In [ ]:
# Query demographics and phenotype covariates
pheno_sql = """
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        p_sex_concept.concept_name as sex_at_birth,
        p_race_concept.concept_name as race,
        p_ethnicity_concept.concept_name as ethnicity
    FROM
        `""" + os.environ['WORKSPACE_CDR'] + """.person` person
    LEFT JOIN `""" + os.environ['WORKSPACE_CDR'] + """.concept` p_sex_concept
        ON person.sex_at_birth_concept_id = p_sex_concept.concept_id
    LEFT JOIN `""" + os.environ['WORKSPACE_CDR'] + """.concept` p_race_concept
        ON person.race_concept_id = p_race_concept.concept_id
    LEFT JOIN `""" + os.environ['WORKSPACE_CDR'] + """.concept` p_ethnicity_concept
        ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id
    WHERE
        person.PERSON_ID IN (
            SELECT DISTINCT person_id
            FROM `""" + os.environ['WORKSPACE_CDR'] + """.cb_search_person`
            WHERE has_lr_whole_genome_variant = 1
        )
"""

print('Querying BigQuery...')
demo_df = pd.read_gbq(
    pheno_sql,
    dialect='standard',
    use_bqstorage_api=('BIGQUERY_STORAGE_API_ENABLED' in os.environ),
    progress_bar_type='tqdm_notebook'
)
demo_df['person_id'] = demo_df['person_id'].astype(str)
print('Demographic rows:', len(demo_df))


### Step 3: Merge exact age at blood draw and build phenotype dataframe

In [ ]:
# Load biosample collection date from genomic_metrics.tsv
QC_METRICS_PATH = 'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv'

qc_df = pd.read_csv(
    QC_METRICS_PATH, sep='\t',
    storage_options={'requester_pays': True, 'project': os.environ['GOOGLE_PROJECT']}
)

if 'sample_name' in qc_df.columns:
    qc_df = qc_df.rename(columns={'sample_name': 'person_id'})
elif 'research_id' in qc_df.columns:
    qc_df = qc_df.rename(columns={'research_id': 'person_id'})
qc_df['person_id'] = qc_df['person_id'].astype(str)

clinical = demo_df.merge(
    qc_df[['person_id', 'biosample_collection_date']],
    on='person_id', how='inner'
).copy()

clinical['date_of_birth'] = pd.to_datetime(clinical['date_of_birth'], utc=True)
clinical['biosample_collection_date'] = pd.to_datetime(
    clinical['biosample_collection_date'], utc=True
)
clinical['Age'] = (
    clinical['biosample_collection_date'] - clinical['date_of_birth']
).dt.days / 365.25

clinical['sex_binary'] = (clinical['sex_at_birth'] == 'Male').astype(int)

clinical = clinical[['person_id', 'Age', 'sex_at_birth', 'sex_binary', 'race', 'ethnicity']]
clinical = clinical.dropna(subset=['Age']).drop_duplicates('person_id')
print('Clinical participants with usable age:', len(clinical))


### Step 4: Compute Epigenetic Age Deviation (EAD)

In [ ]:
# Load per-sample predictions from GCS
# Each cell type has a CSV at cross_reference_ont.../per_sample_results_{celltype}.csv
# For simplicity, we rebuild EAD from the matched summary_df

# EAD = pred_age - true_age (raw acceleration)
# Residual EAD: regress out chronological age (and age^2 for Myeloid)

CELL_TYPES = ['Myeloid', 'Lymphoid', 'T_Cell', 'B_Cell', 'NK_Cell', 'Monocyte', 'Granulocyte']

def compute_residual_ead(pred_age, true_age, quadratic=False):
    """Regress pred_age on true_age (+ true_age^2 if quadratic), return residuals."""
    X = pd.DataFrame({'age': true_age})
    X['const'] = 1.0
    if quadratic:
        X['age2'] = true_age ** 2
    result = sm.OLS(pred_age, X[sorted(X.columns)]).fit()
    return result.resid.values, result

print('EAD computation functions defined.')


### Step 5: OLS regression: EAD ~ phenotype + covariates

In [ ]:
def fit_ead_association(df, phenotype_col, is_myeloid=False):
    """Regress EAD on a binary phenotype, controlling for age (and age^2 for Myeloid)."""
    df = df.copy().dropna(subset=['EAD', phenotype_col, 'Age', 'sex_binary'])
    if len(df) < 20:
        return None

    X = pd.DataFrame({
        'const':      1.0,
        'age':        df['Age'],
        phenotype_col: df[phenotype_col],
        'sex':        df['sex_binary'],
    })
    if is_myeloid:
        X['age2'] = df['Age'] ** 2

    result = sm.OLS(df['EAD'], X).fit()
    return {
        'phenotype':   phenotype_col,
        'beta':        float(result.params[phenotype_col]),
        'se':          float(result.bse[phenotype_col]),
        'p_value':     float(result.pvalues[phenotype_col]),
        'n':           int(len(df)),
        'ci_lo':       float(result.conf_int().loc[phenotype_col, 0]),
        'ci_hi':       float(result.conf_int().loc[phenotype_col, 1]),
    }

print('OLS helper defined.')


### Step 6: Delta-AIC quadratic vs linear age model

In [ ]:
# Delta-AIC: compare quadratic vs linear age adjustment for Myeloid clock
# (Justifies using age^2 term in Myeloid EAD regressions)

def delta_aic_quadratic_vs_linear(pred_age, true_age):
    """Compare AIC of linear vs quadratic age model for Myeloid clock."""
    X_linear = sm.add_constant(true_age)
    X_quad   = sm.add_constant(np.column_stack([true_age, true_age**2]))

    m_linear = sm.OLS(pred_age, X_linear).fit()
    m_quad   = sm.OLS(pred_age, X_quad).fit()

    return {
        'AIC_linear':    m_linear.aic,
        'AIC_quadratic': m_quad.aic,
        'delta_AIC':     m_linear.aic - m_quad.aic,
        'R2_linear':     m_linear.rsquared,
        'R2_quadratic':  m_quad.rsquared,
    }

print('Delta-AIC helper defined.')
print('Run cells below after loading per-sample prediction data.')


### Step 7: Forest plot of phenotype associations across clocks

In [ ]:
def draw_forest_plot(results_df, title='EAD Phenotype Associations', figsize=(8, 6)):
    """Forest plot: beta with 95% CI per phenotype x clock."""
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
        'pdf.fonttype': 42, 'ps.fonttype': 42,
        'figure.dpi': 150, 'savefig.dpi': 300,
    })

    phenotypes = results_df['phenotype'].unique()
    cell_types = results_df['cell_type'].unique()

    colors = plt.cm.tab10(np.linspace(0, 1, len(cell_types)))
    color_map = dict(zip(cell_types, colors))

    fig, ax = plt.subplots(figsize=figsize, facecolor='white')
    y_pos = 0
    y_ticks, y_labels = [], []

    for pheno in phenotypes:
        sub = results_df[results_df['phenotype'] == pheno].copy()
        for _, row in sub.iterrows():
            color = color_map.get(row['cell_type'], 'gray')
            ax.errorbar(
                x=row['beta'],
                y=y_pos,
                xerr=[[row['beta'] - row['ci_lo']], [row['ci_hi'] - row['beta']]],
                fmt='o',
                color=color,
                capsize=3,
                markersize=5,
            )
            star = ' *' if row['p_value'] < 0.05 else ''
            y_ticks.append(y_pos)
            y_labels.append(f"{row['cell_type']}{star}")
            y_pos += 1
        ax.axhline(y_pos - 0.5, color='lightgray', linewidth=0.5)
        ax.text(-0.01, y_pos - len(sub)/2 - 0.5, pheno, ha='right',
                fontsize=9, fontweight='bold', transform=ax.get_yaxis_transform())
        y_pos += 1

    ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=8)
    ax.set_xlabel('Effect size (years EAD per unit phenotype)', fontsize=10)
    ax.set_title(title, fontweight='bold', fontsize=11)
    legend_patches = [mpatches.Patch(color=color_map[ct], label=ct) for ct in cell_types]
    ax.legend(handles=legend_patches, fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    return fig

print('Forest plot function defined.')
print('Load per-sample prediction results and call draw_forest_plot(results_df) to render.')
